# AI Reading Comprehension: Deep Learning Text Generation
## Fine-Tuning T5 for Question Generation
This notebook is designed to be run on **Google Colab** using the T4 GPU.
It fulfills the project requirement of training a Neural Network sequence-to-sequence model and evaluating it using BLEU, ROUGE, and METEOR.

**Instructions:**
1. Go to `Runtime > Change runtime type` and select **T4 GPU**.
2. Upload your `train.csv` and `val.csv` files to the Colab files panel on the left.
3. Click `Runtime > Run all`.


# New Section

In [1]:
!pip install transformers datasets evaluate rouge_score nltk accelerate meteor


In [2]:
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq
import evaluate
import numpy as np
import nltk

nltk.download('wordnet')
nltk.download('punkt')


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

## 1. Load the RACE Dataset
*Note: We load a subset of the data (5,000 rows) so that the notebook finishes in a reasonable time on Colab. You can increase this if you want to train longer!*


In [9]:
# Load dataset
train_df = pd.read_csv('train_small.csv').head(5000)
val_df = pd.read_csv('val_small.csv').head(500)

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

print(f"Training on {len(train_dataset)} examples.")


Training on 5000 examples.


## 2. Preprocess Data for T5
T5 requires a specific prompt format. We prepend `generate question:` to the article.


In [10]:
model_name = "t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def preprocess_function(examples):
    # Prefix the input with a task description for T5
    inputs = ["generate question: " + str(doc) for doc in examples["article"]]
    model_inputs = tokenizer(inputs, max_length=512, truncation=True)

    # Setup targets
    labels = tokenizer(text_target=[str(q) for q in examples["question"]], max_length=128, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_train = train_dataset.map(preprocess_function, batched=True)
tokenized_val = val_dataset.map(preprocess_function, batched=True)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

## 3. Define Evaluation Metrics (BLEU, ROUGE, METEOR)
Your teacher requested these exact metrics for text generation.


In [16]:
rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")
meteor = evaluate.load("meteor")

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]

    # Replace -100 in BOTH preds and labels before decoding!
    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Compute metrics
    result_rouge = rouge.compute(predictions=decoded_preds, references=decoded_labels)

    # BLEU expects references to be a list of lists
    bleu_refs = [[ref] for ref in decoded_labels]
    result_bleu = bleu.compute(predictions=decoded_preds, references=bleu_refs)

    result_meteor = meteor.compute(predictions=decoded_preds, references=decoded_labels)

    return {
        "rougeL": result_rouge["rougeL"],
        "bleu": result_bleu["bleu"],
        "meteor": result_meteor["meteor"],
    }


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


## 4. Train the Neural Network (with Checkpointing)
This block automatically saves checkpoints to your Google Drive folder so you don't lose progress if Colab crashes.


In [17]:
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

# Define training arguments WITH CHECKPOINTING
training_args = Seq2SeqTrainingArguments(
    output_dir="./t5_rc_checkpoints",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    save_total_limit=3,
    save_strategy="epoch",
    num_train_epochs=3,
    predict_with_generate=True,
    fp16=True,  # Fast training on GPU
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Starting Deep Learning Training...")
trainer.train()


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Starting Deep Learning Training...


Epoch,Training Loss,Validation Loss,Rougel,Bleu,Meteor
1,2.841243,2.164156,0.179672,0.021681,0.141589
2,2.291232,2.020198,0.185700,0.025396,0.150112
3,2.188494,1.985298,0.183435,0.024842,0.147136


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1875, training_loss=2.3791525065104167, metrics={'train_runtime': 371.632, 'train_samples_per_second': 40.363, 'train_steps_per_second': 5.045, 'total_flos': 1582230247833600.0, 'train_loss': 2.3791525065104167, 'epoch': 3.0})

## 5. View Final Metrics!
Once training is complete, the final metrics will be printed above. You can screenshot those metrics (BLEU, ROUGE, METEOR) and put them directly in your Final Report!
